In [1]:
import numpy as np
from pathlib import Path
import torch
from sklearn.metrics import root_mean_squared_error, r2_score
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
from NN_model import ImprovedNN


def set_freeze_mode(model, freeze_level=0):
    """
    freeze_level:
        0 = train all layers
        1 = freeze first hidden block
        2 = freeze first two hidden blocks
        3 = freeze first three hidden blocks (if present)
    """
    block_size = 4  # [Linear, BatchNorm, ReLU, Dropout] per hidden layer

    # Unfreeze everything first
    for p in model.parameters():
        p.requires_grad = True

    if freeze_level == 0:
        print("Freeze Level 0: all layers trainable")
        return

    # How many blocks actually exist?
    n_blocks_total = len(model.network) // block_size  # e.g., 3 blocks for [256,128,64]
    n_blocks = min(freeze_level, n_blocks_total)
    print(f"Freeze Level {freeze_level}: freezing {n_blocks} block(s)")

    for b in range(n_blocks):
        start = b * block_size
        for i in range(start, start + 2):  # [Linear, BatchNorm]
            layer = model.network[i]
            for p in layer.parameters():
                p.requires_grad = False



def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, 
                   fold_idx, checkpoint_dir, checkpoint_tracking, is_final=False):
        
        
        # Calculate val predictions
        model.eval()
        all_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds = model(xb).cpu().numpy()
                all_preds.append(preds)
        preds_val = np.concatenate(all_preds)
        
        # Calculate performance metrics from val predictions
        checkpoint_rmse = root_mean_squared_error(y_val, preds_val)
        checkpoint_r2 = r2_score(y_val, preds_val)
        checkpoint_q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
        
        # Create checkpoint filename
        if is_final:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
        else:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"
        
        checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename
        
        # Save the checkpoint
        checkpoint_data = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 'val_loss': val_loss, 'rmse': checkpoint_rmse, 'r2': checkpoint_r2, 'q2': checkpoint_q2,
            'fold_idx': fold_idx, 'is_final': is_final}
        torch.save(checkpoint_data, checkpoint_path_full)
        
        # Record info for spreadsheet
        checkpoint_info = {'Fold': fold_idx, 'Epoch': epoch, 'Checkpoint_Filename': checkpoint_filename, 'Checkpoint_Path': str(checkpoint_path_full),
            'Train_Loss': round(train_loss, 6), 'Val_Loss': round(val_loss, 6), 'RMSE': round(checkpoint_rmse, 6), 'R2': round(checkpoint_r2, 6), 
            'Q2': round(checkpoint_q2, 6), 'Is_Final': is_final}
        checkpoint_tracking.append(checkpoint_info)
        
        checkpoint_type = "Final" if is_final else "Regular"
        print(f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - RMSE: {checkpoint_rmse:.4f}")
        return True


# since RMSE Loss is not in PyTorch, we define it here using MSELoss

class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):  

        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps      # eps: a small constant to avoid sqrt(0) or division by zero;  to prevent potential numerical instability or "division by zero" like issues if the MSE happens to be exactly zero 

    def forward(self, y_pred, y_true):
        mse = self.mse(y_pred, y_true)
        rmse = torch.sqrt(mse + self.eps)
        return rmse
    

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    

def evaluate_fold_TL(
    trial, fold_idx,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    hidden_layers, dropout_rate,
    learning_rate, weight_decay, batch_size,
    freeze_level,                # 0,1,2,3 → how many feature blocks to freeze
    baseline_ckpt,               # path to medium-range baseline .pth
    max_epochs = 10**9,
    patience   = 30,
    min_delta  = 0.0,
    X_test_scaled=None, y_test=None,
    save_checkpoints=False, checkpoint_dir="checkpoints", save_every_n_epochs=15
):
    """
    Transfer-learning fold trainer using a SINGLE learning rate (no param groups).
    Expects pre-scaled numpy arrays (no scaling here).

    Returns:
        rmse, r2, q2, model, train_losses, val_losses, stop_epoch
    """
    device = torch.device("cpu")
    print(f"Fold {fold_idx}: TL on cpu | freeze={freeze_level} | lr={learning_rate:g}")

    # checkpoint bookkeeping
    checkpoint_tracking = []
    fold_checkpoint_dir = None
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # tensors/loaders (inputs are already scaled)
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
    y_train_tensor = torch.tensor(y_train,        dtype=torch.float32, device=device)
    X_val_tensor   = torch.tensor(X_val_scaled,   dtype=torch.float32, device=device)
    y_val_tensor   = torch.tensor(y_val,          dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size, shuffle=False, num_workers=0
    )

    # --- model: same arch as baseline; load baseline weights ---
    model = ImprovedNN(
        input_size = X_train_scaled.shape[1],
        hidden_layers = hidden_layers,
        dropout_rate  = dropout_rate
    ).to(device)

    state = torch.load(baseline_ckpt, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"], strict=True)
    else:
        model.load_state_dict(state, strict=True)

    # --- freeze policy ---
    set_freeze_mode(model, freeze_level)

    # --- optimizer: SINGLE LR over all trainable params ---
    optimizer = optim.AdamW(
        (p for p in model.parameters() if p.requires_grad),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # loss & scheduler & early stopping (same semantics as baseline)
    criterion = RMSELoss()  # your existing class
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    train_losses, val_losses = [], []
    stop_epoch = None

    # --- training loop ---
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)                 # shape [B,] from your ImprovedNN
            loss  = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = model(xb)
                loss  = criterion(preds, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # save periodic checkpoints
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader, fold_idx,
                fold_checkpoint_dir, checkpoint_tracking, is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader, fold_idx,
                    fold_checkpoint_dir, checkpoint_tracking, is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train {train_loss:.4f} | Val {val_loss:.4f} | ES {early_stopper.counter}/{patience}")

    # restore best
    model.load_state_dict(best_state)
    model.eval()

    # optional: export the checkpoint-tracking spreadsheet
    if save_checkpoints and checkpoint_tracking:
        df_checkpoints = pd.DataFrame(checkpoint_tracking)
        spreadsheet_file = fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints.csv"
        df_checkpoints.to_csv(spreadsheet_file, index=False)
        print(f"[Fold {fold_idx}] Checkpoint spreadsheet saved: {spreadsheet_file}")
        print(f"[Fold {fold_idx}] Total checkpoints saved: {len(checkpoint_tracking)}")

    # final metrics on validation
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_preds.append(model(xb).cpu().numpy())
    preds_val = np.concatenate(all_preds)

    rmse = root_mean_squared_error(y_val, preds_val)
    r2   = r2_score(y_val, preds_val)
    q2   = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)

    return rmse, r2, q2, model, train_losses, val_losses, stop_epoch


In [2]:
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN  # your model
from pathlib import Path
import json, torch

# 0) Load ALREADY-SCALED high-MW data (same feature order as baseline)
df_high = pd.read_csv("artifacts/final_train_high_plus_low_700_scaled_reclustered.csv")    # <- already scaled


TARGET_COL = "MP"

exclude = {"SMILES", TARGET_COL, "MW", "MP_category_default", "Structure_Cluster"}
num_cols = df_high.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

X = df_high[feature_cols].to_numpy(np.float32)  # <-- already scaled
y = df_high[TARGET_COL].to_numpy(np.float32)
y_strat = df_high["Structure_Cluster"].astype(str).to_numpy()

# 1) Fix folds once

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr, va) for tr, va in skf.split(X, y_strat)]


BASELINE_CKPT = Path("artifacts/int_Mw_best_models/low_Mw_best_fold_1.pt")  # Checkpoint or pipeline


/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd, torch, optuna

# --- scenarios: name, vector (for your notes), freeze_level used by evaluate_fold_TL ---

HIDDEN_LAYERS = [256,128,64]   # must match baseline arch
N_TRIALS      = 20

OUT_ROOT = Path("artifacts/TL_mixed_700_cv")   # under the artifacts folder
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_one_scenario(tag, freeze_vec, freeze_level):
    print(f"\n=== Scenario: {tag} | freeze={freeze_vec} (level={freeze_level}) ===")
    SCEN_OUT = OUT_ROOT / tag
    (SCEN_OUT / "trials").mkdir(parents=True, exist_ok=True)

    def objective_tl_fixed(trial):
        # fixed freeze level; tune the rest
        learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
        weight_decay  = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])
        dropout_rate  = trial.suggest_float("dropout_rate", 0.2, 0.5)

        trial_dir = SCEN_OUT / "trials" / f"trial_{trial.number:04d}"
        trial_dir.mkdir(parents=True, exist_ok=True)

        fold_metrics, rmses = [], []
        for fold_idx, (tr_idx, va_idx) in enumerate(folds):
            X_tr, y_tr = X[tr_idx], y[tr_idx]
            X_va, y_va = X[va_idx], y[va_idx]

            rmse, r2, q2, model, *_ = evaluate_fold_TL(
                trial=trial,
                fold_idx=fold_idx,
                X_train_scaled=X_tr, y_train=y_tr,
                X_val_scaled=X_va,   y_val=y_va,
                hidden_layers=HIDDEN_LAYERS, dropout_rate=dropout_rate,
                learning_rate=learning_rate, weight_decay=weight_decay,
                batch_size=batch_size, freeze_level=freeze_level,
                baseline_ckpt=BASELINE_CKPT,
                max_epochs=10**6, patience=30, min_delta=0.0,
                save_checkpoints=False
            )

            ckpt_path = trial_dir / f"fold_{fold_idx}_best.pth"
            torch.save(model.state_dict(), ckpt_path)

            fold_metrics.append({
                "fold": fold_idx,
                "rmse": float(rmse),
                "r2":   float(r2),
                "q2":   float(q2),
                "checkpoint": str(ckpt_path)
            })
            rmses.append(rmse)

        summary = {
            "scenario": tag,
            "freeze_vector": freeze_vec,
            "freeze_level": freeze_level,
            "trial_number": trial.number,
            "params": {
                "learning_rate": learning_rate,
                "weight_decay":  weight_decay,
                "batch_size":    batch_size,
                "dropout_rate":  dropout_rate,
                "hidden_layers": HIDDEN_LAYERS
            },
            "avg_rmse": float(np.mean(rmses)),
            "folds":    fold_metrics
        }
        with open(trial_dir / "summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        return float(np.mean(rmses))

    # -- HPO
    study = optuna.create_study(direction="minimize")
    study.optimize(objective_tl_fixed, n_trials=N_TRIALS, gc_after_trial=True)

    # save study artifacts
    joblib.dump(study, SCEN_OUT / "study.joblib")
    study.trials_dataframe().to_csv(SCEN_OUT / "trials.csv", index=False)
    with open(SCEN_OUT / "best_params.json","w") as f:
        json.dump(study.best_params, f, indent=2)
    with open(SCEN_OUT / "best_value.txt","w") as f:
        f.write(f"{study.best_value:.6f}\n")
    print(f"[{tag}] Best avg RMSE: {study.best_value:.4f}")
    print(f"[{tag}] Best params:  {study.best_params}")

    # -- Final per-fold retrain with best params (to produce clean fold models + metrics)
    best = study.best_params
    FINAL_DIR = SCEN_OUT / "final_fold_models"
    FINAL_DIR.mkdir(parents=True, exist_ok=True)

    rows = []
    for fold_idx, (tr_idx, va_idx) in enumerate(folds):
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_va, y_va = X[va_idx], y[va_idx]

        rmse, r2, q2, model, *_ = evaluate_fold_TL(
            trial=None,
            fold_idx=fold_idx,
            X_train_scaled=X_tr, y_train=y_tr,
            X_val_scaled=X_va,   y_val=y_va,
            hidden_layers=HIDDEN_LAYERS,
            dropout_rate=best["dropout_rate"],
            learning_rate=best["learning_rate"],
            weight_decay=best["weight_decay"],
            batch_size=best["batch_size"],
            freeze_level=freeze_level,
            baseline_ckpt=BASELINE_CKPT,
            max_epochs=10**6, patience=30, min_delta=0.0,
            save_checkpoints=False
        )

        ckpt = FINAL_DIR / f"fold_{fold_idx}_best.pth"
        torch.save(model.state_dict(), ckpt)
        rows.append({"fold": fold_idx, "rmse": float(rmse), "r2": float(r2), "q2": float(q2), "checkpoint": str(ckpt)})

    cv_df = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
    cv_df.to_csv(SCEN_OUT / "cv_final_metrics.csv", index=False)

    best_row = cv_df.iloc[0]
    manifest = {
        "scenario": tag,
        "freeze_vector": freeze_vec,
        "freeze_level": freeze_level,
        "best_fold": int(best_row["fold"]),
        "checkpoint": best_row["checkpoint"],
        "hidden_layers": HIDDEN_LAYERS,
        "best_params": best
    }
    with open(SCEN_OUT / "manifest.json","w") as f:
        json.dump(manifest, f, indent=2)

    print(f"[{tag}] Best fold: {manifest['best_fold']} → {manifest['checkpoint']}")
    return study, cv_df, manifest


# ---------- RUN ALL THREE ----------
SCENARIOS = [
    ("no_freeze",        [0,0,0], 0),
    ("freeze_fc1",       [1,0,0], 1),
    ("freeze_fc1_fc2",   [1,1,0], 2),
]

results = {}
for tag, vec, lvl in SCENARIOS:
    study, cv_df, manifest = run_one_scenario(tag, vec, lvl)
    results[tag] = {"best": study.best_value, "manifest": manifest}
print("\nSummary:", json.dumps(results, indent=2))

[I 2026-01-20 10:47:56,851] A new study created in memory with name: no-name-60cdbe61-e12c-496d-abbd-36095b9e3b0d



=== Scenario: no_freeze | freeze=[0, 0, 0] (level=0) ===
Fold 0: TL on cpu | freeze=0 | lr=1.44222e-05
Freeze Level 0: all layers trainable


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/1863631501.py:195: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(baseline_ckpt, map_locat

[Fold 0] Epoch    1 | Train 96.4807 | Val 96.1657 | ES 0/30
[Fold 0] Epoch   50 | Train 87.2895 | Val 84.7340 | ES 0/30
[Fold 0] Epoch  100 | Train 84.5429 | Val 81.7018 | ES 2/30
[Fold 0] Epoch  150 | Train 83.2649 | Val 81.7315 | ES 15/30
[Fold 0] Early stopping at epoch 165 (best Val Loss: 79.5283)
Fold 1: TL on cpu | freeze=0 | lr=1.44222e-05
Freeze Level 0: all layers trainable
[Fold 1] Epoch    1 | Train 97.1935 | Val 84.6098 | ES 0/30


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/1863631501.py:195: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(baseline_ckpt, map_locat

...

[Fold 7] Epoch   50 | Train 57.9574 | Val 40.4121 | ES 26/30
[Fold 7] Early stopping at epoch 54 (best Val Loss: 39.7266)
Fold 8: TL on cpu | freeze=2 | lr=0.000376994
Freeze Level 2: freezing 2 block(s)
[Fold 8] Epoch    1 | Train 94.0641 | Val 78.1120 | ES 0/30


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/1863631501.py:195: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(baseline_ckpt, map_locat

[Fold 8] Epoch   50 | Train 59.9962 | Val 47.4018 | ES 1/30
[Fold 8] Epoch  100 | Train 59.3948 | Val 46.6070 | ES 9/30
[Fold 8] Early stopping at epoch 121 (best Val Loss: 46.1025)
Fold 9: TL on cpu | freeze=2 | lr=0.000376994
Freeze Level 2: freezing 2 block(s)
[Fold 9] Epoch    1 | Train 94.6765 | Val 82.2925 | ES 0/30


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/1863631501.py:195: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(baseline_ckpt, map_locat

[Fold 9] Epoch   50 | Train 58.0301 | Val 42.7427 | ES 6/30
[Fold 9] Early stopping at epoch 74 (best Val Loss: 41.2132)
[freeze_fc1_fc2] Best fold: 7 → artifacts/TL_mixed_700_cv/freeze_fc1_fc2/final_fold_models/fold_7_best.pth

Summary: {
  "no_freeze": {
    "best": 46.567664337158206,
    "manifest": {
      "scenario": "no_freeze",
      "freeze_vector": [
        0,
        0,
        0
      ],
      "freeze_level": 0,
      "best_fold": 7,
      "checkpoint": "artifacts/TL_mixed_700_cv/no_freeze/final_fold_models/fold_7_best.pth",
      "hidden_layers": [
        256,
        128,
        64
      ],
      "best_params": {
        "learning_rate": 0.0009533722333095898,
        "weight_decay": 1.3703185254383669e-05,
        "batch_size": 32,
        "dropout_rate": 0.43572627832045097
      }
    }
  },
  "freeze_fc1": {
    "best": 46.35166244506836,
    "manifest": {
      "scenario": "freeze_fc1",
      "freeze_vector": [
        1,
        0,
        0
      ],
      "freez

In [4]:
def load_best_models_for_scenarios(
    root_dir="artifacts/TL_mixed_700_cv",
    scenarios=("no_freeze", "freeze_fc1", "freeze_fc1_fc2"),
):
    root_dir = Path(root_dir)
    models = {}
    manifests = {}

    for tag in scenarios:
        manifest_path = root_dir / tag / "manifest.json"
        cv_metrics_path = root_dir / tag / "cv_final_metrics.csv"

        # Load manifest
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
        manifests[tag] = manifest

        # Load best RMSE from cv_final_metrics.csv
        cv_df = pd.read_csv(cv_metrics_path)
        best_row = cv_df.sort_values("rmse").iloc[0]
        best_rmse = float(best_row["rmse"])

        # Load checkpoint
        ckpt_path = Path(best_row["checkpoint"]).resolve()
        state = torch.load(ckpt_path, map_location="cpu")

        # Build model
        input_size = state["network.0.weight"].shape[1]
        hidden_layers = manifest["hidden_layers"]
        dropout_rate = manifest["best_params"]["dropout_rate"]

        model = ImprovedNN(
            input_size=input_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
        )
        model.load_state_dict(state)
        model.eval()

        models[tag] = model

        print(f"Loaded model for scenario '{tag}'")
        print(f"  └─ Best fold checkpoint: {ckpt_path}")
        print(f"  └─ Best RMSE: {best_rmse:.4f}\n")

    return models, manifests

models, manifests = load_best_models_for_scenarios(
    root_dir="artifacts/TL_mixed_700_cv",
    scenarios=["no_freeze", "freeze_fc1", "freeze_fc1_fc2"]
)

Loaded model for scenario 'no_freeze'
  └─ Best fold checkpoint: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/TL_mixed_700_cv/no_freeze/final_fold_models/fold_7_best.pth
  └─ Best RMSE: 39.5040

Loaded model for scenario 'freeze_fc1'
  └─ Best fold checkpoint: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/TL_mixed_700_cv/freeze_fc1/final_fold_models/fold_7_best.pth
  └─ Best RMSE: 37.6130

Loaded model for scenario 'freeze_fc1_fc2'
  └─ Best fold checkpoint: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/TL_mixed_700_cv/freeze_fc1_fc2/final_fold_models/fold_7_best.pth
  └─ Best RMSE: 40.1311



/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/1541890903.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location="

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from NN_model import ImprovedNN  # adjust if your import differs


# PATHS
# --------------------
BASE = Path.cwd()  # SDL5_MP/transfer_learning
CKPT_PTH = BASE / "artifacts/TL_mixed_700_cv/freeze_fc1/final_fold_models/fold_7_best.pth" ####################################
TEST_SCALED_CSV = BASE / "artifacts/final_test_df_with_rdkit_scaled.csv"
OUT_PRED_CSV = BASE / "artifacts/test_eval_TL_mixed_700_predictions.csv"


# MODEL PARAMETERS
HIDDEN_LAYERS = [256, 128, 64]
DROPOUT_RATE = 0.25337976695847986   # IMPORTANT: use best_params dropout_rate


# --------------------
# Load scaled test set
# --------------------
df_test = pd.read_csv(TEST_SCALED_CSV)

rdkit_cols = [c for c in df_test.columns if c not in ["SMILES", "exp MP"]]

X_test = df_test[rdkit_cols].astype(np.float32).values
y_true = df_test["exp MP"].astype(float).values
smiles = df_test["SMILES"].astype(str).values


print("Test rows:", len(df_test))
print("RDKit features:", X_test.shape[1])


# --------------------
# Recreate model + load checkpoint
# --------------------
device = torch.device("cpu")  # change to "cuda" if desired/available

model = ImprovedNN(input_size=X_test.shape[1], hidden_layers=HIDDEN_LAYERS, dropout_rate=DROPOUT_RATE).to(device)
loaded = torch.load(CKPT_PTH, map_location=device)

# Support both formats:
# 1) checkpoint dict with "model_state_dict"
# 2) raw state_dict
if isinstance(loaded, dict) and "model_state_dict" in loaded:
    model.load_state_dict(loaded["model_state_dict"])
else:
    model.load_state_dict(loaded)

model.eval()

# --------------------
# Predict
# --------------------
X_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)

with torch.no_grad():
    y_pred = model(X_tensor).squeeze().detach().cpu().numpy().astype(float)


# --------------------
# Evaluate
# --------------------
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
mae  = float(mean_absolute_error(y_true, y_pred))
r2   = float(r2_score(y_true, y_pred))

print("\n=== TEST METRICS ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R^2 : {r2:.4f}")


# --------------------
# Save per-compound predictions
# --------------------
out_df = pd.DataFrame({
    "SMILES": smiles,
    "exp MP": y_true,
    "pred MP": y_pred,
    "error": y_pred - y_true,
    "abs_error": np.abs(y_pred - y_true),
})

OUT_PRED_CSV.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(OUT_PRED_CSV, index=False)
print(f"\nSaved predictions -> {OUT_PRED_CSV}")


Test rows: 1961
RDKit features: 202

=== TEST METRICS ===
RMSE: 53.0314
MAE : 39.3043
R^2 : 0.6523

Saved predictions -> /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/artifacts/test_eval_TL_mixed_700_predictions.csv


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_88279/718181011.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded = torch.load(CKPT_PTH, map_location=de

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

PRED_CSV = "artifacts/test_eval_TL_mixed_700_predictions.csv"

df = pd.read_csv(PRED_CSV)

print(df.head())
print("Total samples:", len(df))

                            SMILES  exp MP     pred MP       error   abs_error
0                        BrB(Br)Br   -46.0   54.235825  100.235825  100.235825
1            O=S(c1ccccc1)c1ccccc1    71.0   36.332924  -34.667076   34.667076
2                        CC1(C)CC1  -109.0    9.178401  118.178401  118.178401
3            CCCCCCCCCCS(=O)(=O)Cl    32.0   35.880146    3.880146    3.880146
4  N#CC(Nc1nc(NC2CC2)nc(n1)Cl)(C)C   170.0  220.871689   50.871689   50.871689
Total samples: 1961


In [7]:
MP_THRESHOLD = 250.0   # <-- example, change if needed

df_low  = df[df["exp MP"] < MP_THRESHOLD]
df_high = df[df["exp MP"] >= MP_THRESHOLD]

print("Low MP samples :", len(df_low))
print("High MP samples:", len(df_high))

Low MP samples : 1882
High MP samples: 79


In [8]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


rmse_low = rmse(df_low["exp MP"], df_low["pred MP"])
rmse_high = rmse(df_high["exp MP"], df_high["pred MP"])
rmse_all = rmse(df["exp MP"], df["pred MP"])

print("\n=== RMSE by group ===")
print(f"Overall RMSE : {rmse_all:.4f}")
print(f"Low MP RMSE  : {rmse_low:.4f}")
print(f"High MP RMSE : {rmse_high:.4f}")



=== RMSE by group ===
Overall RMSE : 53.0314
Low MP RMSE  : 52.9708
High MP RMSE : 54.4555
